# Can TGLF's trapped-particle and collision knobs bring its converged KBM growth rate down to GS2's?

At WIDTH 0.4, TGLF's $\gamma$ converges in NBASIS to a value 0.3–0.5 dex above GS2 (`basis_resolution_rule`). The eigenmode is right and
the growth rate is wrong, so the suspect is the physics model. TGLF bounce-averages the particles whose bounce length fits inside the mode
(`tglf_geometry.f90:740-783`):

$$k_\parallel=\frac{2\pi}{L_y\sqrt2\,\mathrm{WIDTH}},\qquad y_b=\min\!\left(L_y,\ \frac{\pi\,\mathrm{THETA\_TRAPPED}}{k_\parallel}\right),\qquad f_t=\sqrt{1-\frac{B_{\min}}{B(y_b)}},$$

and under SAT_RULE 2 (whose presets force XNU_MODEL 3, WDIA_TRAPPED 1) each species gets
$k_\parallel\to k_\parallel/\mathrm{THETA\_TRAPPED}+\omega_{\rm dia}c_{dt}$, $\omega_{\rm dia}=\lvert k_y\,a/L_n\rvert/v_s$,
$c_{dt}=3\,\mathrm{WDIA\_TRAPPED}\,(1-f_{t0}^2)$, $f_{t0}=\sqrt{1-B_{\min}/B_{\max}}$. Trapped particles are not Landau damped, so a larger
$f_t$ means less damping.

**Runs** (TGLF only; catalogue section "TGLF `kbm_free`: trapped-particle and collision knobs at WIDTH 0.4"): the 738 GS2-KBMs, WIDTH 0.4,
SAT_RULE 2, FILTER $-1.5$, NXGRID 32, `FIND_WIDTH` F, MHD rule off, $A_\parallel$ and $B_\parallel$ on, NBASIS 16 and 24.
- Baseline: the qp5.37 leaves `fx_fm1p5_w0p4000_nb{16,24}`.
- One knob changed per leaf `tk_<knob>_nb<N>`: THETA_TRAPPED, XNU_FACTOR, PARK, and `sat1` = SAT_RULE 1 with UNITS CGYRO, the only
  reachable XNU_MODEL 2 + WDIA_TRAPPED 0 point (`tglf_startup.f90:45-63` overwrites both from SAT_RULE).

**Example KBMs** (choices are the user's to change): `sample_0818` plus 5 per database, drawn from GS2 data only (seed 0): one from each
GS2-$\gamma$ quartile, the $\phi$-extent quartile assigned by a random permutation, plus one more at random. The remaining 717 are the
out-of-sample set.

In [ ]:
import os
import re
from multiprocessing import get_context
from pathlib import Path
import warnings
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import NullLocator, StrMethodFormatter
from pyrokinetics import PyroHypercube
from pyrokinetics.diagnostics.extent import Extent

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir())
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)
warnings.filterwarnings("ignore")
data_root = Path(os.path.expandvars(os.environ["GK_DATA_ROOT"]))     # the WORK tree
plots_dir = ROOT / "Plots" / "TGLF_GFTM_KBM_PAPER" / "tglf_trapped_knobs"
plots_dir.mkdir(parents=True, exist_ok=True)
n_cpu = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))

cases = ["SPR-045", "M1", "R4", "R3"]
case_colour = dict(zip(cases, ["#4477aa", "#ee6677", "#228833", "#ccbb44"]))
width = 0.4
baseline = {"THETA_TRAPPED": 0.7, "XNU_FACTOR": 1.0, "PARK": 1.0, "presets": "SAT2"}
# knob -> {value: leaf tag}; the baseline value maps to the qp5.37 leaf
knobs = {"THETA_TRAPPED": {0.1: "tt0p1", 0.3: "tt0p3", 0.5: "tt0p5", 0.7: None, 0.9: "tt0p9", 1.0: "tt1", 3.5355: "tt3p5355"},
         "XNU_FACTOR": {0.1: "xnuf0p1", 0.3: "xnuf0p3", 1.0: None, 3.0: "xnuf3", 10.0: "xnuf10"},
         "PARK": {0.5: "park0p5", 1.0: None, 2.0: "park2"},
         # SAT_RULE's presets (tglf_startup.f90:45-63): SAT2 = XNU_MODEL 3, WDIA_TRAPPED 1; SAT1 = XNU_MODEL 2, WDIA_TRAPPED 0 (UNITS CGYRO in both)
         "presets": {"SAT2": None, "SAT1": "sat1", "SAT1, THETA_TRAPPED 3.5355": "sat1tt3p5355"}}   # the last at NBASIS 24 only


def leaf_path(case, tag, nbasis):
    name = f"fx_fm1p5_w0p4000_nb{nbasis}" if tag is None else f"tk_{tag}_nb{nbasis}"
    return data_root / "TGLF/Runs/LATIN_HYPERCUBE" / case / "kbm_free" / name


def extent_95(squared_field):
    # pyro's 95 % extent, sample by sample; GS2's theta grid is NaN outside each sample's own points, so NaNs are dropped first
    return np.array([float(Extent._compute(row.dropna("theta").expand_dims("sample"), 0.95)[0][0]) if row.notnull().any() else np.nan
                     for row in squared_field.transpose("sample", "theta")])

## GS2 reference for the 738 KBMs, and the 21 example modes

In [ ]:
gs2_rows = []
for case in cases:
    leaf = data_root / "GS2/Runs/LATIN_HYPERCUBE" / case / "kbm_free"
    gs2_scan = PyroHypercube(pyroscan_json=leaf / "pyroscan.json", load_base_pyro=True)
    gs2_scan.load_gk_output(netcdf_file=leaf / "pyroscan.nc")
    population = [str(name) for name in PyroHypercube(pyroscan_json=leaf_path(case, None, 24) / "pyroscan.json", load_base_pyro=True).pyro_dict]
    gs2_output = gs2_scan.gk_output.data.set_index(sample="sample_name").sel(sample=population)
    gs2_rows.append(pd.DataFrame(dict(case=case, sample=population,
                                      gamma_gs2=gs2_output.growth_rate.pint.dequantify().values.ravel(),
                                      omega_gs2=gs2_output.mode_frequency.pint.dequantify().values.ravel(),
                                      gamma_units_gs2=re.sub(r"_pyroscan_base\d+", "", str(gs2_output.growth_rate.data.units)),
                                      phi_extent_gs2=extent_95(gs2_output.phi_squared.pint.dequantify().squeeze("kx")),
                                      apar_extent_gs2=extent_95(gs2_output.apar_squared.pint.dequantify().squeeze("kx")))))
gs2 = pd.concat(gs2_rows, ignore_index=True)
assert len(gs2) == 738, len(gs2)

rng = np.random.default_rng(0)
gs2["example"] = (gs2.case == "SPR-045") & (gs2["sample"] == "sample_0818")
for case in cases:
    pool = gs2[(gs2.case == case) & ~gs2.example].copy()
    pool["gamma_quartile"] = pd.qcut(pool.gamma_gs2, 4, labels=False)
    pool["extent_quartile"] = pd.qcut(pool.phi_extent_gs2, 4, labels=False)
    for gamma_quartile, extent_quartile in enumerate(rng.permutation(4)):
        cell = pool[(pool.gamma_quartile == gamma_quartile) & (pool.extent_quartile == extent_quartile)]
        if cell.empty:   # no sample in that cell: the nearest extent quartile in the same gamma quartile
            same_gamma = pool[pool.gamma_quartile == gamma_quartile]
            cell = same_gamma[(same_gamma.extent_quartile - extent_quartile).abs() == (same_gamma.extent_quartile - extent_quartile).abs().min()]
        gs2.loc[rng.choice(cell.index), "example"] = True
    rest = gs2[(gs2.case == case) & ~gs2.example]
    gs2.loc[rng.choice(rest.index), "example"] = True
assert gs2.example.sum() == 21
gs2[gs2.example].sort_values(["case", "gamma_gs2"])

## TGLF runs: dominant mode of every leaf
IBRANCH $-1$ returns the modes ordered by growth rate, so mode 0 is the dominant one (asserted on every leaf).

In [ ]:
def load_leaf(spec):
    case, knob, value, tag, nbasis = spec
    scan = PyroHypercube(pyroscan_json=leaf_path(case, tag, nbasis) / "pyroscan.json", load_base_pyro=True)
    scan.load_gk_output(netcdf_file=leaf_path(case, tag, nbasis) / "pyroscan.nc")
    output = scan.gk_output.data.set_index(sample="sample_name")
    growth_rate = output.growth_rate.pint.dequantify()
    assert (growth_rate.fillna(-np.inf).argmax("mode") == 0).all()
    dominant = output.isel(mode=0)  # mode0-guard-allow: IBRANCH -1, argmax == 0 asserted above
    return pd.DataFrame(dict(case=case, knob=knob, value=value, nbasis=nbasis, sample=[str(name) for name in output.sample.values],
                             gamma_units=re.sub(r"_pyroscan_base\d+", "", str(dominant.growth_rate.data.units)),
                             gamma=dominant.growth_rate.pint.dequantify().values,
                             omega=dominant.mode_frequency.pint.dequantify().values,
                             phi_extent=extent_95(abs(dominant.phi.pint.dequantify().squeeze("kx")) ** 2),
                             apar_extent=extent_95(abs(dominant.apar.pint.dequantify().squeeze("kx")) ** 2)))


specs = [(case, knob, value, tag, nbasis) for case in cases for knob, values in knobs.items() for value, tag in values.items() for nbasis in (16, 24)
         if (tag is not None or knob == "THETA_TRAPPED") and not (tag == "sat1tt3p5355" and nbasis == 16)]     # the baseline leaf is read once, under THETA_TRAPPED, and copied to the other knobs
with get_context("fork").Pool(n_cpu) as pool:
    runs = pd.concat(pool.map(load_leaf, specs), ignore_index=True)
base_runs = runs[(runs.knob == "THETA_TRAPPED") & (runs.value == baseline["THETA_TRAPPED"])]
runs = pd.concat([runs] + [base_runs.assign(knob=knob, value=baseline[knob]) for knob in knobs if knob != "THETA_TRAPPED"], ignore_index=True)
runs = runs.merge(gs2, on=["case", "sample"], validate="many_to_one")
runs["value"] = runs.value.astype(object)
assert (runs.groupby(["knob", "value", "nbasis"], sort=False).size() == 738).all()
assert (runs.gamma_units == runs.gamma_units_gs2).all(), "TGLF and GS2 growth rates are not in the same normalisation"
runs["found"] = (runs.gamma > 0) & (runs.omega > 0)      # a KBM-like mode: unstable, ion direction
for quantity in ("gamma", "omega", "phi_extent", "apar_extent"):
    runs[f"{quantity}_ratio"] = (runs[quantity] / runs[f"{quantity}_gs2"]).where(runs.found)
runs.groupby(["knob", "value", "nbasis"], sort=False).found.mean().unstack().round(3)

## Step 1: is the baseline converged? NBASIS 16 against 24
Converged: $\lvert\gamma_{16}/\gamma_{24}-1\rvert\le 5\,\%$ (the tolerance is a stated default). The same check is made for every knob value.

In [ ]:
pair = runs.pivot_table(index=["knob", "value", "case", "sample", "example"], columns="nbasis", values=["gamma", "omega"], sort=False).reset_index()
pair["gamma_change"] = abs(pair[("gamma", 16)] / pair[("gamma", 24)] - 1)
pair["omega_change"] = abs(pair[("omega", 16)] / pair[("omega", 24)] - 1)
convergence = pair.groupby(["knob", "value"], sort=False).apply(lambda group: pd.Series({
    "21 examples within 5 %": (group[group.example].gamma_change <= 0.05).mean(),
    "738 within 5 %": (group.gamma_change <= 0.05).mean(),
    "738 median |gamma_16/gamma_24 - 1|": group.gamma_change.median(),
    "738 median |omega_16/omega_24 - 1|": group.omega_change.median()}))
convergence.to_csv(plots_dir / "convergence_nb16_nb24.csv")
convergence.round(3)

In [ ]:
base24 = runs[(runs.knob == "THETA_TRAPPED") & (runs.value == 0.7) & (runs.nbasis == 24)]
base_table = base24[base24.example].merge(pair[(pair.knob == "THETA_TRAPPED") & (pair.value == 0.7)][["case", "sample", "gamma_change"]].droplevel(1, axis=1),
                                          on=["case", "sample"])
base_table[["case", "sample", "gamma_gs2", "gamma", "gamma_change", "gamma_ratio", "omega_ratio", "phi_extent_ratio", "apar_extent_ratio"]].sort_values(["case", "gamma_gs2"]).round(3)

## TGLF's trapped fraction
TGLF prints the electron $f_t$ it used on the `ft:` line of `out.tglf.run` (`tglf.f90:143` prints the global `ft`, which
`tglf_eigensolver.f90:113` overwrites with `fts(1)`). Under SAT_RULE 2 that includes the electron diamagnetic term; under SAT_RULE 1 it does not,
so the SAT_RULE 1 leaf at THETA_TRAPPED $\sqrt2/0.4=3.5355$ gives each mode's full geometric $f_t$ as TGLF computes it. Pyrokinetics does not read
this line and the run directories are on scratch only, so it is read there, for the 21 examples. The ion $f_t$ is not written anywhere.

A recomputation from pyro's Miller geometry was tried and did not reproduce TGLF (`sample_0818`, SAT_RULE 1: geometric $f_t$ 0.683 against TGLF's
0.648, and 0.338 against 0.226 at THETA_TRAPPED 0.7), so it is not used.

In [ ]:
def printed_ft(case, sample, leaf):
    run_file = Path(os.environ["GYRO_DATA_OUTPUT"]) / "TGLF/Runs/LATIN_HYPERCUBE" / case / "kbm_free" / leaf / sample / "out.tglf.run"
    return float(re.search(r"ft:\s*(\S+)", run_file.read_text()).group(1))


examples = gs2[gs2.example].copy()
examples["ft_geometric"] = [printed_ft(case, sample, "tk_sat1tt3p5355_nb24") for case, sample in zip(examples.case, examples["sample"])]
examples["ft_sat1"] = [printed_ft(case, sample, "tk_sat1_nb24") for case, sample in zip(examples.case, examples["sample"])]
for theta_trapped, tag in knobs["THETA_TRAPPED"].items():
    leaf = "fx_fm1p5_w0p4000_nb24" if tag is None else f"tk_{tag}_nb24"
    examples[f"ft_{theta_trapped:g}"] = [printed_ft(case, sample, leaf) for case, sample in zip(examples.case, examples["sample"])]
runs = runs.merge(examples[["case", "sample", "ft_geometric", "ft_sat1"] + [f"ft_{value:g}" for value in knobs["THETA_TRAPPED"]]], on=["case", "sample"], how="left")
examples[["case", "sample", "ft_geometric", "ft_sat1", "ft_0.7", "ft_0.1", "ft_1", "ft_3.5355"]].sort_values(["case", "sample"]).round(3)

## Step 2: one knob at a time
Per knob: $\gamma_{\rm TGLF}/\gamma_{\rm GS2}$, $\omega$ ratio and both extent ratios against the knob value at NBASIS 24, one thin line per example mode
(coloured by database), the median of the 21 in black, and the median of all 738 dashed. NBASIS 16 medians are dotted (spot check).

In [ ]:
panels = [("gamma_ratio", r"$\gamma_{\rm TGLF}/\gamma_{\rm GS2}$"), ("omega_ratio", r"$\omega_{\rm TGLF}/\omega_{\rm GS2}$"),
          ("phi_extent_ratio", r"$\phi$ extent ratio"), ("apar_extent_ratio", r"$A_\parallel$ extent ratio")]
for knob, values in knobs.items():
    data = runs[runs.knob == knob]
    fig, axes = plt.subplots(1, 4, figsize=(20, 4.6), constrained_layout=True)
    for ax, (column, label) in zip(axes, panels):
        examples = data[data.example & (data.nbasis == 24)]
        for (case, sample), mode in examples.groupby(["case", "sample"]):
            mode = mode.sort_values("value")
            ax.plot(mode.value, mode[column], color=case_colour[case], lw=0.8, alpha=0.7, marker=".", label=case)
        for nbasis, style in ((24, "-"), (16, ":")):
            ax.plot(*zip(*data[data.example & (data.nbasis == nbasis)].groupby("value")[column].median().items()), "k" + style, lw=2.5, label=f"median of 21, NBASIS {nbasis}")
        ax.plot(*zip(*data[data.nbasis == 24].groupby("value")[column].median().items()), "k--", lw=1.5, label="median of 738, NBASIS 24")
        ax.axhline(1, color="grey", lw=0.8)
        ax.set_yscale("log"); ax.set_xlabel(knob, fontsize=16); ax.set_title(label, fontsize=16); ax.tick_params(axis="both", labelsize=13)
        if knob == "presets":
            ax.tick_params(axis="x", labelrotation=20, labelsize=10)
        else:
            ax.set_xscale("log"); ax.xaxis.set_major_formatter(StrMethodFormatter("{x:g}")); ax.xaxis.set_minor_locator(NullLocator())
            ax.set_xticks(sorted(values))
    handles, labels = axes[0].get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    axes[0].legend(unique.values(), unique.keys(), fontsize=9)
    fig.savefig(plots_dir / f"knob_{knob}.pdf"); fig.savefig(plots_dir / f"knob_{knob}.png", dpi=150)
    plt.show()

In [ ]:
summary = runs[runs.nbasis == 24].groupby(["knob", "value"], sort=False).apply(lambda group: pd.Series({
    "median gamma ratio (21)": group[group.example].gamma_ratio.median(),
    "median gamma ratio (738)": group.gamma_ratio.median(),
    "median omega ratio (738)": group.omega_ratio.median(),
    "median phi extent ratio (738)": group.phi_extent_ratio.median(),
    "median apar extent ratio (738)": group.apar_extent_ratio.median(),
    "gamma within 20 % of GS2 (738)": (abs(group.gamma_ratio - 1) <= 0.2).mean(),
    "found (738)": group.found.mean()}))
summary.to_csv(plots_dir / "knob_summary_nb24.csv")
summary.round(3)

## Mechanism: $\gamma$ against the trapped fraction
The THETA_TRAPPED scan moves only the trapped fractions. $\gamma$ ratio against TGLF's electron $f_t$ per example mode (SAT_RULE 2, NBASIS 24),
with each mode's geometric $f_t$ (SAT_RULE 1, THETA_TRAPPED 3.5355) marked by a cross at its $\gamma$ ratio in that leaf.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
data = runs[(runs.knob == "THETA_TRAPPED") & (runs.nbasis == 24) & runs.example].copy()
data["ft"] = [data.loc[index, f"ft_{value:g}"] for index, value in data.value.items()]
geometric = runs[(runs.knob == "presets") & (runs.value == "SAT1, THETA_TRAPPED 3.5355") & runs.example]
for (case, sample), mode in data.groupby(["case", "sample"]):
    mode = mode.sort_values("ft")
    ax.plot(mode.ft, mode.gamma_ratio, color=case_colour[case], marker=".", lw=0.8, label=case)
for row in geometric.itertuples():
    ax.plot(row.ft_geometric, row.gamma_ratio, "x", color=case_colour[row.case], ms=8)
ax.axhline(1, color="grey", lw=0.8); ax.set_yscale("log")
ax.set_xlabel(r"electron $f_t$ (TGLF)", fontsize=16); ax.set_ylabel(r"$\gamma_{\rm TGLF}/\gamma_{\rm GS2}$", fontsize=16); ax.tick_params(labelsize=13)
handles, labels = ax.get_legend_handles_labels()
ax.legend(dict(zip(labels, handles)).values(), dict(zip(labels, handles)).keys(), fontsize=10)
fig.savefig(plots_dir / "gamma_vs_ft.pdf"); fig.savefig(plots_dir / "gamma_vs_ft.png", dpi=150)
plt.show()

## Validation format: $\gamma$ within $x$ of GS2 against knob value
The `*_gamma_gs2_within_*` fraction of `basis_resolution_rule`, with the knob value on the x-axis: all 738, the 21 examples (in-sample) and the
717 others (out-of-sample), at NBASIS 24 and 16.

In [ ]:
for tolerance in (0.1, 0.2):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), constrained_layout=True, sharey=True)
    for ax, knob in zip(axes, ["THETA_TRAPPED", "XNU_FACTOR", "PARK"]):
        data = runs[runs.knob == knob]
        for nbasis, colour in ((24, "#4477aa"), (16, "#ee6677")):
            for subset, style, label in ((slice(None), "-", "738"), (data.example, "--", "21 (in-sample)"), (~data.example, ":", "717 (out-of-sample)")):
                fraction = data[subset][data[subset].nbasis == nbasis].groupby("value").gamma_ratio.apply(lambda ratio: (abs(ratio - 1) <= tolerance).mean())
                ax.plot(fraction.index, fraction.values, style, color=colour, marker="o", ms=3, label=f"NBASIS {nbasis}, {label}")
        ax.set_xscale("log"); ax.xaxis.set_major_formatter(StrMethodFormatter("{x:g}")); ax.xaxis.set_minor_locator(NullLocator())
        ax.set_xticks(sorted(knobs[knob])); ax.set_ylim(0, 1); ax.set_xlabel(knob, fontsize=16); ax.tick_params(axis="both", labelsize=13)
        ax.set_title(f"TGLF, $\\gamma$ within {tolerance:.0%} of GS2", fontsize=15)
    axes[0].set_ylabel("fraction of KBMs", fontsize=16); axes[0].legend(fontsize=9)
    stem = f"TGLF_knobs_gamma_gs2_within_{round(100 * tolerance)}pct"
    fig.savefig(plots_dir / f"{stem}.pdf"); fig.savefig(plots_dir / f"{stem}.png", dpi=150)
    plt.show()